# Predicting the Source of a News Headline (Doc2Vec + ANN)

Given a news headline, predict which publisher wrote it.

**Steps:** load data → keep 6 publishers → clean text → train/test split → Doc2Vec (headline → 100 numbers) → ANN (numbers → publisher) → check accuracy

Dataset: [UCI News Aggregator (Kaggle)](https://www.kaggle.com/datasets/uciml/news-aggregator-dataset). Put `uci-news-aggregator.csv` next to this notebook.

In [1]:
import re
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score, average_precision_score)
from sklearn.preprocessing import label_binarize

## Step 1: Load data and keep publishers with 2,000–3,000 headlines

In [2]:
df = pd.read_csv("uci-news-aggregator.csv")
news = df[["TITLE", "PUBLISHER"]].dropna()   # URL is dropped: it contains the publisher name

counts = news["PUBLISHER"].value_counts()
publishers = counts[(counts > 2000) & (counts < 3000)].index
news = news[news["PUBLISHER"].isin(publishers)].reset_index(drop=True)

print("Headlines:", len(news))
news["PUBLISHER"].value_counts()

Headlines: 13751


PUBLISHER
Huffington Post     2455
Businessweek        2395
Contactmusic.com    2334
Daily Mail          2254
NASDAQ              2228
Examiner.com        2085
Name: count, dtype: int64

## Step 2: Clean the text (lowercase, remove punctuation, split into words)

In [3]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9$%' ]", " ", text)
    return text.split()

news["words"] = news["TITLE"].apply(clean)
news[["TITLE", "words"]].head()

,TITLE,words
0,Plosser: Fed May Have to Accelerate Tapering Pace,"[plosser, fed, may, have, to, accelerate, tape..."
1,Forex - Pound drops to one-month lows against ...,"[forex, pound, drops, to, one, month, lows, ag..."
2,"Euro Anxieties Wane as Bunds Top Treasuries, S...","[euro, anxieties, wane, as, bunds, top, treasu..."
3,Noyer Says Strong Euro Creates Unwarranted Eco...,"[noyer, says, strong, euro, creates, unwarrant..."
4,EBay CEO Donahoe's Pay Drops by 53% to $13.8 M...,"[ebay, ceo, donahoe's, pay, drops, by, 53%, to..."


## Step 3: Split into train (80%) and test (20%)

In [4]:
train, test = train_test_split(news, test_size=0.2, stratify=news["PUBLISHER"], random_state=42)
print("Train:", len(train), " Test:", len(test))

Train: 11000  Test: 2751


## Step 4: Doc2Vec — turn every headline into 100 numbers

- `dm=0` → PV-DBOW mode (fast, good for short text)
- `dbow_words=1` → also learns word meanings at the same time, which gives better headline vectors
- Trained only on the training headlines

In [5]:
documents = [TaggedDocument(words, [i]) for i, words in enumerate(train["words"])]

d2v = Doc2Vec(dm=0, dbow_words=1, window=5, vector_size=100, min_count=2, epochs=40, workers=4, seed=42)
d2v.build_vocab(documents)
d2v.train(documents, total_examples=d2v.corpus_count, epochs=d2v.epochs)

X_train = np.array([d2v.infer_vector(w, epochs=40) for w in train["words"]])
X_test = np.array([d2v.infer_vector(w, epochs=40) for w in test["words"]])
print("Each headline is now", X_train.shape[1], "numbers")

Each headline is now 100 numbers


## Step 5: ANN — learn which numbers belong to which publisher

100 inputs → 256 neurons → 128 neurons → one output per publisher (softmax).
ReLU activation, Adam optimizer, early stopping to avoid overfitting.

In [6]:
scaler = StandardScaler().fit(X_train)
labels = LabelEncoder().fit(train["PUBLISHER"])

ann = MLPClassifier(hidden_layer_sizes=(256, 128), activation="relu", solver="adam",
                    alpha=1e-3, early_stopping=True, max_iter=300, random_state=42)
ann.fit(scaler.transform(X_train), labels.transform(train["PUBLISHER"]))
print("Training stopped after", ann.n_iter_, "epochs")

Training stopped after 15 epochs


## Step 6: Evaluate on test headlines the model never saw

Same metrics as a fraud-detection model, calculated for each publisher (one publisher vs the rest) and then averaged:

| Metric | Meaning |
|---|---|
| Accuracy | % of headlines with the correct publisher |
| Precision | When the model says publisher X, how often it is right |
| Recall | Of all real publisher-X headlines, how many it found |
| F1 | Balance of precision and recall |
| False alarm rate | Of headlines NOT from X, how many were wrongly given to X |
| ROC-AUC | How well the model ranks the right publisher above the others (0.5 = random, 1 = perfect) |
| PR-AUC | Area under the precision-recall curve |
| Top-2 accuracy | % of headlines where the right publisher is in the model's top 2 guesses |

In [7]:
X_test_s = scaler.transform(X_test)
y_test = labels.transform(test["PUBLISHER"])
probs = ann.predict_proba(X_test_s)
pred = probs.argmax(axis=1)
names = labels.classes_

y_onehot = label_binarize(y_test, classes=range(len(names)))
top2 = np.argsort(probs, axis=1)[:, -2:]

cm = confusion_matrix(y_test, pred)
fp = cm.sum(axis=0) - np.diag(cm)
tn = cm.sum() - cm.sum(axis=0) - cm.sum(axis=1) + np.diag(cm)
false_alarm = fp / (fp + tn)

overall = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision (macro)": precision_score(y_test, pred, average="macro"),
    "Recall (macro)": recall_score(y_test, pred, average="macro"),
    "F1 (macro)": f1_score(y_test, pred, average="macro"),
    "False alarm rate (macro)": false_alarm.mean(),
    "ROC-AUC (macro, one-vs-rest)": roc_auc_score(y_test, probs, multi_class="ovr"),
    "PR-AUC (macro)": average_precision_score(y_onehot, probs, average="macro"),
    "Top-2 accuracy": np.mean([y in t for y, t in zip(y_test, top2)]),
}
pd.Series(overall).round(3).to_frame("Score")

,Score
Accuracy,0.572
Precision (macro),0.572
Recall (macro),0.570
F1 (macro),0.565
False alarm rate (macro),0.086
"ROC-AUC (macro, one-vs-rest)",0.860
PR-AUC (macro),0.603
Top-2 accuracy,0.765


### Per publisher

In [8]:
per_pub = pd.DataFrame({
    "Precision": precision_score(y_test, pred, average=None),
    "Recall": recall_score(y_test, pred, average=None),
    "F1": f1_score(y_test, pred, average=None),
    "False alarm rate": false_alarm,
    "ROC-AUC": roc_auc_score(y_onehot, probs, average=None),
    "PR-AUC": average_precision_score(y_onehot, probs, average=None),
}, index=names).round(3)
per_pub.sort_values("F1", ascending=False)

,Precision,Recall,F1,False alarm rate,ROC-AUC,PR-AUC
NASDAQ,0.760,0.850,0.802,0.052,0.970,0.897
Businessweek,0.677,0.699,0.688,0.070,0.930,0.751
Contactmusic.com,0.518,0.632,0.569,0.120,0.880,0.554
Daily Mail,0.598,0.392,0.474,0.052,0.819,0.539
Huffington Post,0.411,0.454,0.431,0.142,0.769,0.397
Examiner.com,0.470,0.393,0.428,0.079,0.791,0.479


### Confusion matrix (rows = real publisher, columns = predicted)

In [9]:
pd.DataFrame(cm, index=names, columns=names)

,Businessweek,Contactmusic.com,Daily Mail,Examiner.com,Huffington Post,NASDAQ
Businessweek,335,3,10,9,50,72
Contactmusic.com,8,295,35,67,61,1
Daily Mail,30,98,177,35,101,10
Examiner.com,28,80,32,164,98,15
Huffington Post,43,92,42,69,223,22
NASDAQ,51,1,0,5,10,379


## Step 7: Try a new headline

In [10]:
def predict_source(headline):
    vector = scaler.transform([d2v.infer_vector(clean(headline), epochs=40)])
    probs = ann.predict_proba(vector)[0]
    top3 = np.argsort(probs)[::-1][:3]
    return [(names[i], round(float(probs[i]), 2)) for i in top3]

predict_source("Stocks fall as Fed hints at faster tapering")

[('NASDAQ', 0.54), ('Businessweek', 0.45), ('Huffington Post', 0.01)]